In [4]:
from pulp import *
from itertools import combinations, permutations

model= LpProblem('Dinner_party_seating_optimiser', LpMaximize)

#variable feeds
adjacent_pairs = {(0,1) : 1, (1,2): 2, (2,3):1, (3,4):1, 
                  (4,5):2, (5,0):1}
names = ['A', 'B', 'C', 'D','E','F']
perm_names = list(permutations(names,2))
pairs = list(combinations(names, 2))
happiness_dict = {(g1,g2): 1 for (g1,g2) in combinations(names, 2)}
happiness_dict[('A','B')] = 10


#decision variables
x = LpVariable.dicts("seat", [(g,s,) for g in names for s 
                              in range(6)], cat='Binary')
z = LpVariable.dicts("pair", [(g1,g2,s1,s2) for (g1,g2) in 
                              perm_names for (s1,s2) in
                              adjacent_pairs], 
                              cat='Binary')

#obj function
model += (lpSum(happiness_dict[g1,g2]*adjacent_pairs[s1,s2]
                *(z[(g1,g2,s1,s2)] + z[(g2,g1,s1,s2)]) for 
                (g1,g2) in pairs for (s1,s2) in 
                adjacent_pairs))

#constraints
#1. each guest is seated once
for s in range(6):
    model += lpSum(x[(g,s)] for g in names) == 1

#2. each seat is used once
for g in names:
    model += lpSum(x[(g,s)] for s in range(6)) == 1

#3. to linearise z to be binary
for (s1,s2) in adjacent_pairs:
    for (g1,g2) in perm_names:
        model += z[(g1,g2,s1,s2)] <= x[(g1,s1)]
        model += z[(g1,g2,s1,s2)] <= x[(g2,s2)]
        model += z[(g1,g2,s1,s2)] >= (x[(g1,s1)] 
                                      + x[(g2,s2)] -1)

#solve model
model.solve()
guest_seating_dict = {}

for s in range(6):
    for g in names:
        if x[g,s].varValue == 1:
            guest_seating_dict[s] = g

print(guest_seating_dict)

{0: 'F', 1: 'B', 2: 'A', 3: 'D', 4: 'C', 5: 'E'}
